In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt

# === CONFIG ===
base_path = r'/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Hyperparameters
BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-3
IMG_SIZE = 256

# === DATASET CLASS ===
class COCOSegmentationDataset(Dataset):
    def __init__(self, split='train', img_size=IMG_SIZE, transform=None):
        self.split = split
        self.img_size = img_size
        self.transform = transform

        masks_dir = os.path.join(base_path, f"mask/masks_{split}")
        self.mask_files = [f for f in os.listdir(masks_dir) if f.endswith('.png')]
        self.masks_dir = masks_dir

        # Extract image names (remove '_mask.png')
        self.img_names = [f.replace('_mask.png', '.jpg') for f in self.mask_files]
        self.img_dir = os.path.join(base_path, 'raw', 'train2017' if split == 'train' else 'val2017')

        print(f"Loaded {len(self.mask_files)} {split} samples")

    def __len__(self):
        return len(self.mask_files)

    def __getitem__(self, idx):
        # Load image
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        # Load mask
        mask_name = self.mask_files[idx]
        mask_path = os.path.join(self.masks_dir, mask_name)
        mask = Image.open(mask_path).convert('L')  # Grayscale

        # Resize to same size
        image = image.resize((self.img_size, self.img_size))
        mask = mask.resize((self.img_size, self.img_size))

        # Convert to tensors
        image = np.array(image).transpose(2, 0, 1) / 255.0  # HWC -> CHW, normalize
        mask = np.array(mask) / 255.0  # Binary 0/1

        image = torch.FloatTensor(image)
        mask = torch.FloatTensor(mask)

        if self.transform:
            image = self.transform(image)

        return image, mask.unsqueeze(0)  # Add channel dim for mask

# === SIMPLE U-NET MODEL ===
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True)
            )

        # Encoder
        self.enc1 = conv_block(in_channels, 64)
        self.enc2 = conv_block(64, 128)
        self.enc3 = conv_block(128, 256)

        # Decoder
        self.upconv3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = conv_block(256, 128)
        self.upconv2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = conv_block(128, 64)

        self.final_conv = nn.Conv2d(64, out_channels, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(nn.MaxPool2d(2)(e1))
        e3 = self.enc3(nn.MaxPool2d(2)(e2))

        # Decoder
        d2 = self.upconv3(e3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.upconv2(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        out = self.final_conv(d1)
        return self.sigmoid(out)

# === LOSS & METRICS ===
class DiceLoss(nn.Module):
    def __init__(self, smooth=1):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred = pred.contiguous().view(-1)
        target = target.contiguous().view(-1)

        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        return 1 - dice

def iou_score(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    target = (target > 0.5).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + 1e-6) / (union + 1e-6)

# === DATA LOADERS ===
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])

train_dataset = COCOSegmentationDataset('train', transform=train_transform)
val_dataset = COCOSegmentationDataset('val')

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, BATCH_SIZE, shuffle=False, num_workers=2)

# === TRAINING ===
model = UNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)
criterion = DiceLoss()

# Training history
train_losses, val_losses = [], []
train_ious, val_ious = [], []

print("Starting training...")
for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss, train_iou = 0, 0
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')

    for images, masks in train_pbar:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        preds = model(images)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_iou += iou_score(preds, masks).item()

        train_pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'IoU': f'{iou_score(preds, masks).item():.4f}'})

    # Validate
    model.eval()
    val_loss, val_iou = 0, 0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc='Val', leave=False):
            images, masks = images.to(device), masks.to(device)
            preds = model(images)
            loss = criterion(preds, masks)

            val_loss += loss.item()
            val_iou += iou_score(preds, masks).item()

    # Log metrics
    train_losses.append(train_loss / len(train_loader))
    val_losses.append(val_loss / len(val_loader))
    train_ious.append(train_iou / len(train_loader))
    val_ious.append(val_iou / len(val_loader))

    print(f'Epoch {epoch+1}: Train Loss={train_losses[-1]:.4f}, IoU={train_ious[-1]:.4f} | '
          f'Val Loss={val_losses[-1]:.4f}, IoU={val_ious[-1]:.4f}')

    scheduler.step(val_losses[-1])

    # Save best model
    if len(val_ious) == 1 or val_ious[-1] > max(val_ious[:-1]):
        torch.save(model.state_dict(), f'coco_unet_best_{IMG_SIZE}.pth')
        print(f"✓ New best model saved (IoU: {val_ious[-1]:.4f})")

# === VISUALIZATION ===
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.legend()
plt.title('Loss')

plt.subplot(1, 3, 2)
plt.plot(train_ious, label='Train IoU')
plt.plot(val_ious, label='Val IoU')
plt.legend()
plt.title('IoU')

plt.subplot(1, 3, 3)
model.eval()
with torch.no_grad():
    sample_img, sample_mask = next(iter(val_loader))
    sample_img = sample_img[:1].to(device)
    pred = model(sample_img)
    pred_np = pred[0,0].cpu().numpy()
    sample_mask_np = sample_mask[0,0].cpu().numpy()

    plt.imshow(pred_np, cmap='gray')
    plt.title(f'Prediction (IoU: {iou_score(pred, sample_img).item():.3f})')
    plt.axis('off')

plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Training completed! Best model saved as 'coco_unet_best_256.pth'")


Using device: cuda
Loaded 118287 train samples
Loaded 5000 val samples
Starting training...


Epoch 1/20 [Train]:   0%|          | 0/14786 [00:00<?, ?it/s]


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipython-input-4175674593.py", line 47, in __getitem__
    image = Image.open(img_path).convert('RGB')
            ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/PIL/Image.py", line 3513, in open
    fp = builtins.open(filename, "rb")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017/raw/train2017/000000110559.jpg'
